# Analyzing Historical Stock/Revenue Data and Building a Dashboard

## Setup: Install Required Libraries

In [ ]:
!pip install yfinance
!pip install bs4
!pip install nbformat
!pip install matplotlib

## Setup: Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import warnings

# Ignore warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Define Graphing Function

This function is provided. You don't need to modify it. It takes a dataframe with stock data (must contain Date and Close columns), a dataframe with revenue data (must contain Date and Revenue columns), and the name of the stock.

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Stock price
    axes[0].plot(pd.to_datetime(stock_data_specific.Date), stock_data_specific.Close.astype("float"), label="Share Price", color="blue")
    axes[0].set_ylabel("Price ($US)")
    axes[0].set_title(f"{stock} - Historical Share Price")

    # Revenue
    axes[1].plot(pd.to_datetime(revenue_data_specific.Date), revenue_data_specific.Revenue.astype("float"), label="Revenue", color="green")
    axes[1].set_ylabel("Revenue ($US Millions)")
    axes[1].set_xlabel("Date")
    axes[1].set_title(f"{stock} - Historical Revenue")

    plt.tight_layout()
    plt.show()

## Question 1: Use yfinance to Extract Tesla Stock Data

In [ ]:
# Create a Ticker object for Tesla
tesla = yf.Ticker("TSLA")

In [ ]:
# Extract stock data for the maximum available period
tesla_data = tesla.history(period="max")

In [ ]:
# Reset the index and display the first five rows
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2: Use Webscraping to Extract Tesla Revenue Data

In [ ]:
# Download the webpage and save the text of the response
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url).text

In [ ]:
# Parse the html data using BeautifulSoup
soup = BeautifulSoup(html_data, "html5lib")

In [ ]:
# Extract the table with Tesla Revenue and store it in a dataframe named tesla_revenue
tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for row in soup.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")
    date = col[0].text
    revenue = col[1].text
    tesla_revenue = pd.concat(
        [tesla_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})],
        ignore_index=True
    )

tesla_revenue.head()

In [ ]:
# Remove the comma and dollar sign from the Revenue column
tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(r',|\$', "", regex=True)

In [ ]:
# Remove null or empty strings in the Revenue column
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]

In [ ]:
# Display the last five rows of the tesla_revenue dataframe
tesla_revenue.tail()

## Question 3: Use yfinance to Extract GameStop Stock Data

In [ ]:
# Create a Ticker object for GameStop
gme = yf.Ticker("GME")

In [ ]:
# Extract stock data for the maximum available period
gme_data = gme.history(period="max")

In [ ]:
# Reset the index and display the first five rows
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4: Use Webscraping to Extract GME Revenue Data

In [ ]:
# Download the webpage and save the text of the response
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data_2 = requests.get(url).text

In [ ]:
# Parse the html data using BeautifulSoup
soup = BeautifulSoup(html_data_2, "html5lib")

In [ ]:
# Extract the table with GameStop Revenue and store it in a dataframe named gme_revenue
gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for row in soup.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")
    date = col[0].text
    revenue = col[1].text
    gme_revenue = pd.concat(
        [gme_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})],
        ignore_index=True
    )

gme_revenue.head()

In [ ]:
# Remove the comma and dollar sign, and any null or empty strings from the Revenue column
gme_revenue["Revenue"] = gme_revenue['Revenue'].str.replace(r',|\$', "", regex=True)
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue['Revenue'] != ""]

In [ ]:
# Display the last five rows of the gme_revenue dataframe
gme_revenue.tail()

## Question 5: Plot Tesla Stock Graph

In [ ]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

## Question 6: Plot GameStop Stock Graph

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')